In [ ]:
# import packages
import pandas as pd
import numpy as np
import os
import plotly.express as px

%reload_ext autoreload
%autoreload 2

# # Tell python where to look for modules.
import sys
sys.path.append('../../../open-grid-emissions/src/')

import download_data
import load_data
from column_checks import get_dtypes
from filepaths import *
import impute_hourly_profiles
import data_cleaning
import output_data
import emissions
import validation
import gross_to_net_generation
import eia930
import consumed

year = 2021
path_prefix = f"{year}/"

In [ ]:
clean_930_file = (outputs_folder(f"{path_prefix}/eia930/eia930_elec.csv"))

hourly_consumed_calc = consumed.HourlyConsumed(
    clean_930_file,
    path_prefix,
    year,
    small=False,
    skip_outputs=False,
)
hourly_consumed_calc.run()
hourly_consumed_calc.output_results()

In [ ]:
# load all power sector results and concat together

resolution = "hourly"

all_data = []
for ba in os.listdir(results_folder(f"{year}/carbon_accounting/{resolution}/us_units")):
    df = pd.read_csv(results_folder(f"{year}/carbon_accounting/{resolution}/us_units/{ba}"))
    df["ba_code"] = ba.split(".")[0]
    all_data.append(df)

all_data = pd.concat(all_data, axis=0)

In [ ]:
# pivot the data for a specific pollutant
pivoted_data = all_data.pivot(index="datetime_utc", columns="ba_code", values="consumed_co2_rate_lb_per_mwh_for_electricity")
pivoted_data

In [ ]:
pivoted_data[pivoted_data.isna().any(axis=1)]

In [ ]:
pivoted_data.isna().sum().sum()

In [ ]:
px.imshow(pivoted_data, height=800)

In [ ]:
# load all power sector results and concat together

resolution = "hourly"

old_data = []
for ba in os.listdir(results_folder(f"{year}/carbon_accounting/hourly/{year}_carbon_accounting_hourly_us_units/")):
    df = pd.read_csv(results_folder(f"{year}/carbon_accounting/hourly/{year}_carbon_accounting_hourly_us_units/{ba}"))
    df["ba_code"] = ba.split(".")[0]
    old_data.append(df)

old_data = pd.concat(old_data, axis=0)

In [ ]:
# pivot the data for a specific pollutant
pivoted_old_data = old_data.pivot(index="datetime_utc", columns="ba_code", values="consumed_co2_rate_lb_per_mwh_for_electricity")
pivoted_old_data

In [ ]:
pivoted_old_data.isna().sum().sum()

In [ ]:
px.imshow(pivoted_old_data, height=800)